# Tugas JS06 ANN — Benchmark Exact NN, Annoy, HNSW, dan FAISS IVF

Notebook ini membandingkan waktu komputasi empat metode pencarian tetangga terdekat pada dataset acak 10.000 vektor berdimensi 128:

| Metode | Tipe |
|---|---|
| Exact NN | brute force dengan `sklearn.NearestNeighbors` |
| Annoy | forest random projection |
| HNSW | graph berlapis (`hnswlib`) |
| FAISS IVF | inverted file index (`faiss`) |

Setiap metode dijalankan pada data yang sama dan diukur waktunya, lalu hasilnya dirangkum dalam tabel.

## 1. Instalasi Library

In [8]:
!pip install -q annoy faiss-cpu hnswlib

## 2. Import Library

In [9]:
import numpy as np
import time
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from annoy import AnnoyIndex
import hnswlib
import faiss
from sklearn.preprocessing import StandardScaler

## 3. Membuat Dataset

Dataset dibuat secara acak dengan `numpy.random.seed(42)` agar hasil dapat direproduksi. Data kemudian distandarisasi menggunakan `StandardScaler`.

In [10]:
# -------------------------------
# Contoh dataset kecil untuk testing
# -------------------------------
np.random.seed(42)
n_samples = 10000   # jumlah database vector
d = 128             # dimensi
X = np.random.random((n_samples, d)).astype('float32')

# Standarisasi fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype(np.float32)

k = 10  # jumlah nearest neighbors

## 4. Exact NN (Brute Force)

Exact NN mencari tetangga terdekat dengan membandingkan query terhadap seluruh data. Waktu yang dicatat mencakup proses `fit` dan `kneighbors`.

In [11]:
# -------------------------------
# Exact NN (brute-force)
# -------------------------------
start = time.time()
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(X_scaled)
dist_exact, idx_exact = nn.kneighbors(X_scaled)
time_exact = time.time() - start
print(f"Exact NN done in {time_exact:.3f} s")

Exact NN done in 1.339 s


## 5. Annoy

Annoy membangun forest dari random projection dengan 10 tree. Waktu yang dicatat mencakup proses `add_item`, `build`, dan seluruh query.

In [12]:
# -------------------------------
# Annoy
# -------------------------------
start = time.time()
f = X_scaled.shape[1]
index_annoy = AnnoyIndex(f, 'euclidean')
for i, v in enumerate(X_scaled):
    index_annoy.add_item(i, v)
index_annoy.build(10)
idx_annoy = [index_annoy.get_nns_by_vector(v, k) for v in X_scaled]
time_annoy = time.time() - start
print(f"Annoy done in {time_annoy:.3f} s")

Annoy done in 0.634 s


## 6. HNSW

HNSW membangun graph berlapis dengan `M=16` dan `ef_construction=200`, lalu melakukan query dengan `ef=200`. Waktu yang dicatat mencakup build dan seluruh query.

In [13]:
# -------------------------------
# HNSW
# -------------------------------
start = time.time()
p_hnsw = hnswlib.Index(space='l2', dim=d)
p_hnsw.init_index(max_elements=n_samples, ef_construction=200, M=16)
p_hnsw.add_items(X_scaled)
p_hnsw.set_ef(200)
idx_hnsw, _ = p_hnsw.knn_query(X_scaled, k=k)
time_hnsw = time.time() - start
print(f"HNSW done in {time_hnsw:.3f} s")

HNSW done in 5.346 s


## 7. FAISS IVF

FAISS IVF membagi data menjadi `nlist=100` cluster dan memeriksa `nprobe=10` cluster terdekat saat query. Waktu yang dicatat mencakup `train`, `add`, dan seluruh query.

In [14]:
# -------------------------------
# FAISS IVF
# -------------------------------
start = time.time()
quantizer = faiss.IndexFlatL2(d)
index_faiss = faiss.IndexIVFFlat(quantizer, d, 100, faiss.METRIC_L2)
index_faiss.train(X_scaled)
index_faiss.add(X_scaled)
index_faiss.nprobe = 10
_, idx_faiss = index_faiss.search(X_scaled, k)
time_faiss = time.time() - start
print(f"FAISS IVF done in {time_faiss:.3f} s")

FAISS IVF done in 0.609 s


## 8. Ringkasan Waktu

Tabel berikut merangkum waktu komputasi setiap metode dari output cell di atas.

In [15]:
# -------------------------------
# Tampilkan ringkasan waktu
# -------------------------------
print("=== Ringkasan Waktu (detik) ===")
print(f"Exact NN : {time_exact:.3f}")
print(f"Annoy    : {time_annoy:.3f}")
print(f"HNSW     : {time_hnsw:.3f}")
print(f"FAISS    : {time_faiss:.3f}")

ringkasan = pd.DataFrame({
    'Metode': ['Exact NN', 'Annoy', 'HNSW', 'FAISS IVF'],
    'Waktu (s)': [time_exact, time_annoy, time_hnsw, time_faiss],
})

display(ringkasan)

=== Ringkasan Waktu (detik) ===
Exact NN : 1.339
Annoy    : 0.634
HNSW     : 5.346
FAISS    : 0.609


,Metode,Waktu (s)
0,Exact NN,1.338962
1,Annoy,0.634222
2,HNSW,5.345786
3,FAISS IVF,0.609038
